# duckdb-kql — advanced features

[![Open in GitHub Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/mmaitre314/duckdb-kql?devcontainer_path=.devcontainer%2Fdemo%2Fdevcontainer.json&quickstart=1)

In [1]:
import duckdb_kql

print("duckdb-kql", duckdb_kql.__version__)

duckdb-kql 0.0.12


In [2]:
con = duckdb_kql.connect()

## Create databases

Use `.create database` to attach new databases. When connecting to in-memory DuckDB, the default database is called `memory`.

In [3]:
duckdb_kql.script(con, '''
    .create database Customers volatile

    .create database Sales volatile

    .show databases | project DatabaseName, IsCurrent, DatabaseAccessMode
''')

[ScriptResult(index=1, line=2, text='.create database Customers volatile', columns=['DatabaseName', 'PersistentPath', 'Created', 'StoresMetadata', 'StoresData'], rows=[('Customers', None, True, True, True)], error=None),
 ScriptResult(index=2, line=4, text='.create database Sales volatile', columns=['DatabaseName', 'PersistentPath', 'Created', 'StoresMetadata', 'StoresData'], rows=[('Sales', None, True, True, True)], error=None),
 ScriptResult(index=3, line=6, text='.show databases | project DatabaseName, IsCurrent, DatabaseAccessMode', columns=['DatabaseName', 'IsCurrent', 'DatabaseAccessMode'], rows=[('Customers', False, 'ReadWrite'), ('Sales', False, 'ReadWrite'), ('memory', True, 'ReadWrite')], error=None)]

In [5]:
duckdb_kql.query(con, database='Customers',
    query='.set-or-replace Customers <| print CustomerId = 1, Name = "Customer-1", Tier = "Free" ')

duckdb_kql.query(con, database='Sales',
    query='.set-or-replace Orders <| print OrderId = 3, CustomerId = 1, Status = "Shipped" ')

duckdb_kql.query(con, '''
    database('Customers').Customers
    | join kind=leftouter (database('Sales').Orders) on CustomerId
    | project-away CustomerId1
''')

┌────────────┬────────────┬─────────┬─────────┬─────────┐
│ CustomerId │    Name    │  Tier   │ OrderId │ Status  │
│   int64    │  varchar   │ varchar │  int64  │ varchar │
├────────────┼────────────┼─────────┼─────────┼─────────┤
│          1 │ Customer-1 │ Free    │       3 │ Shipped │
└────────────┴────────────┴─────────┴─────────┴─────────┘

## Cross-cluster queries

Run cross-cluster queries by mapping the remote databases to local databases.

In [14]:
duckdb_kql.set_clusters({
    ('cluster1.eastus.kusto.windows.net', 'Customers'): 'Customers',
    ('cluster2.westus.kusto.windows.net', 'Sales'): 'Sales',
})

duckdb_kql.query(con, '''
    cluster('cluster1.eastus.kusto.windows.net').database('Customers').Customers
    | join kind=leftouter (cluster('cluster2.westus.kusto.windows.net').database('Sales').Orders) on CustomerId
    | project-away CustomerId1
''')

┌────────────┬────────────┬─────────┬─────────┬─────────┐
│ CustomerId │    Name    │  Tier   │ OrderId │ Status  │
│   int64    │  varchar   │ varchar │  int64  │ varchar │
├────────────┼────────────┼─────────┼─────────┼─────────┤
│          1 │ Customer-1 │ Free    │       3 │ Shipped │
└────────────┴────────────┴─────────┴─────────┴─────────┘

## Macro expansion and entity groups

Register entity groups to run queries calling `macro-expand`.

In [17]:
duckdb_kql.set_entity_groups({
    'MyDatabases': ["database('Customers')"]
})

duckdb_kql.query(con, '''
    macro-expand MyDatabases as db
    (
        db.Customers
        | where Tier == 'Free'
    )
    | count
''')

┌───────┐
│ Count │
│ int64 │
├───────┤
│     1 │
└───────┘